# Example 12.6.4 / 3: Conservatism and validation

Companion notebook for **12.6.4 Robust Counterparts** in the AMPL lab Documentation, section "Conservatism and validation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Conservatism and validation

Report how the set was calibrated and compare nominal objective value, worst-case objective or slack, and held-out performance. Correlated coefficients placed in an independent box can create impossible simultaneous extremes and excessive conservatism.

The price of protection grows with the set. A **budget** Γ allows at most Γ coefficients to deviate
at once; its counterpart adds a variable `z` and one `p[i]` per item, as in the geometry table
below. The run solves for Γ = 0 (nominal), 1 and 2 (the full box):

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                  # decision items
param nominal {ITEMS};                      # nominal coefficient
param deviation {ITEMS} >= 0;               # largest deviation of each coefficient
param value {ITEMS};                        # value per unit of x
param lower {ITEMS};                        # lower bound on x
param upper {ITEMS};                        # upper bound on x
param capacity;                             # right-hand side
param budget >= 0;                          # how many coefficients may deviate together

var x {i in ITEMS} >= lower[i] <= upper[i];
var z >= 0;                                 # protection shared by all items
var p {ITEMS} >= 0;                         # extra protection per item

maximize TotalValue: sum {i in ITEMS} value[i] * x[i];

subject to BudgetCapacity:
    sum {i in ITEMS} nominal[i] * x[i] + budget * z + sum {i in ITEMS} p[i] <= capacity;
subject to Protect {i in ITEMS}:
    z + p[i] >= deviation[i] * abs(x[i]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["a", "b"]
ampl.param["nominal"] = {"a": 2, "b": 1}
ampl.param["deviation"] = {"a": 0.5, "b": 0.5}
ampl.param["value"] = {"a": 3, "b": -1}
ampl.param["lower"] = {"a": 0, "b": -4}
ampl.param["upper"] = {"a": 6, "b": 4}
ampl.param["capacity"] = 10

In [5]:
# run.py
for gamma in (0, 1, 2):
    ampl.param["budget"] = gamma
    ampl.solve(solver="highs", verbose=False)
    print(gamma, ampl.var["x"].to_dict(), round(ampl.obj["TotalValue"].value(), 6))

0 {'a': 6, 'b': -4} 22.0
1 {'a': 5.6, 'b': -4} 20.8
2 {'a': 4.8, 'b': -4} 18.4


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := a b;
param:  nominal  deviation  value  lower  upper :=
  a           2        0.5      3      0      6
  b           1        0.5     -1     -4      4;
param capacity := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
for {gamma in 0..2} {
    let budget := gamma;
    solve;
    printf "%d %g %g %g\n", gamma, x["a"], x["b"], TotalValue;
}

0 6 -4 22
1 5.6 -4 20.8
2 4.8 -4 18.4
